In [1]:
import pandas as pd
import plotly.express as px
from statsmodels.tsa.seasonal import seasonal_decompose
import logging
import datetime
import time

from datetime import datetime

from pyspark.sql.functions import col, datediff
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

import calendar
import pandas as pd
import plotly.express as px
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, col
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from ipywidgets import interact, widgets
from matplotlib.dates import MonthLocator, DateFormatter
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
#from fbprophet import Prophet
from sklearn.metrics import mean_squared_error
import numpy as np
import plotly.graph_objects as go
from pyspark.sql import functions as F
from pyspark.sql.functions import year, month
from sklearn.metrics.pairwise import euclidean_distances
from pyspark.sql.functions import year, month, sum as sum_
from scipy.spatial.distance import euclidean
from prophet import Prophet
from pyspark.sql.functions import col, countDistinct , desc , count , length
from pyspark.sql.functions import split, col, when
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, dayofmonth, last_day ,to_date, lit, isnull
from pyspark.sql.types import IntegerType
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import date_format

from pyspark.sql.functions import col, coalesce, current_date, months_between, round
from pyspark.sql.functions import regexp_replace

import schedule
from pyspark.sql.functions import col, to_timestamp, date_format

import pyspark
import pandas as pd
import boto3
import sagemaker
import sagemaker.feature_store.feature_store as fs
import databricks.connect
from dateutil.relativedelta import relativedelta

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error
from sklearn.metrics import mean_squared_error, root_mean_squared_error

import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import plotly.graph_objects as go
from itertools import product
import random
import statsmodels.api as sm
import os

from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, months_between, current_date,expr
from pyspark.sql.functions import col, substr

import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from databricks.connect import DatabricksSession


# Get spark
spark = databricks.connect.DatabricksSession.builder.getOrCreate()

c:\Users\gusta\AppData\Local\pypoetry\Cache\virtualenvs\ltv-novo-aF9Dy8BA-py3.10\lib\site-packages\pydantic\_internal\_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[02/20/25 12:22:33] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=874263;file://c:\Users\gusta\AppData\Local\pypoetry\Cache\virtualenvs\ltv-novo-aF9Dy8BA-py3.10\lib\site-packages\botocore\credentials.py\credentials.py]8;;\:]8;id=505130;file://c:\Users\gusta\AppData\Local\pypoetry\Cache\virtualenvs\ltv-novo-aF9Dy8BA-py3.10\lib\site-packages\botocore\credentials.py#1278\1278]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\gusta\AppData\Local\sagemaker\sagemaker\config.yaml


In [2]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [3]:
cliente = spark.table("prd.dl_redeok_checkok.tb_cliente") # Tabela com data inici, data final , motivo do bloqueio e max_conexao = next_status , cnpj
revenda = spark.table("prd.dl_redeok_checkok.tb_revenda") # Só usamos para filtrar os contratos tipo revenda
receber = spark.table("prd.dl_redeok_checkok.vw_receber") # Tabela de faturas
estab_df = spark.table("prd.dl_receita_federal.tb_estabelecimento").alias("estab") \
    .withColumn("CNPJ_CPF", concat(col("cnpj_basico"), col("cnpj_ordem"), col("cnpj_dv")))
hierarquia_sdf = spark.table("dev.dw_rok.tb_cnae_hierarquia_comercial").alias("hierarquia")

In [4]:
# Selecionar as colunas desejadas no estab_df
estab_selected = estab_df.select("CNPJ_CPF", "cnae_fiscal_principal","uf","data_inicio_atividade")

# Realizar o join com hierarquia_sdf utilizando a coluna cnae_fiscal_principal e CNAE_RF
resultado_df = estab_selected.join(hierarquia_sdf, estab_selected["cnae_fiscal_principal"] == hierarquia_sdf["CNAE_RF"], "left")

# Suponha que df seja seu DataFrame
#resultado_df = resultado_df.withColumn("ano_nascimento", substr(col("data_inicio_atividade"), 1, 4))

resultado_df = resultado_df.select("CNPJ_CPF", "CNAE_RF", "Hierarquia","uf","data_inicio_atividade")


In [5]:
query=  """WITH 
-- CTE para obter informações básicas dos clientes, como quantidade de contratos, primeiro e último bloqueio, motivo e status de conexão
clientes AS 
(
    SELECT 
          COUNT(1)              AS contratos          -- Contagem total de contratos por documento
        , cgcmf                 AS documento          -- Documento do cliente (CNPJ)
        , MIN(data_cad)         AS data_inicio        -- Data de início do primeiro contrato do cliente
        , MAX(bloqueio)         AS data_final         -- Data do último bloqueio do cliente
        , MAX_BY(max_conexao, bloqueio) AS max_conexao
        , ANY_VALUE(receita)              AS receita         -- Status de conexão mais recente
    FROM 
    (
        -- Seleção dos clientes únicos, eliminando duplicatas e aplicando filtros iniciais
        SELECT DISTINCT
            matriz, codigo, fis_jurid, cgcmf, raz_social, max_conexao, bloqueio, mot_bloqueio,
            sit_cliente, data_cad, vendedor, vlrfatmin, vlr_dispacesso, receita
        FROM 
            prd.dl_redeok_checkok.tb_cliente cli
        WHERE 
            cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')         -- Filtra clientes com status ativos ou pendentes
            AND (cli.matriz = cli.codigo OR falsa = 's')         -- Mantém apenas contas da matriz ou marcadas como falsas
            AND cli.receita IN (9, 10, 46, 58)                   -- Filtra por tipos de receita específico
            AND cli.cgcmf IS NOT NULL                            -- Remove registros sem CNPJ
            AND cli.cgcmf != '23236392000198'                    -- Exclui um CNPJ específico (ROK)
            AND LENGTH(TRIM(cli.cgcmf)) = 14                     -- Garante que o CNPJ tenha 14 caracteres
            AND cli.cgcmf RLIKE '^[0-9]+$'                       -- Garante que o CNPJ seja numérico

            -- Remove registros que fazem parte da tabela de revendas
            AND NOT EXISTS 
            (
                SELECT 1 
                FROM prd.dl_redeok_checkok.tb_revenda rev 
                WHERE cli.codigo = rev.codigo
            )
    ) 
    GROUP BY cgcmf
),

-- CTE para contar a quantidade de boletos gerados por cliente
titulo_cobranca AS 
(
    SELECT 
          documento         AS documento                
        , cliente          AS cliente  
        , COUNT(meses)      AS boletos_gerados        -- Contagem de meses com boletos gerados
    FROM 
    (
    -- Seleção de boletos distintos por cliente e mês
        SELECT DISTINCT
            cli.cgcmf                           AS documento
            , rec.cliente                       AS cliente  
            , DATE_TRUNC('month', rec.emissao)  AS meses     -- Agrupa por mês de emissão do boleto
        FROM 
            prd.dl_redeok_checkok.vw_receber rec
        INNER JOIN prd.dl_redeok_checkok.tb_cliente cli 
            ON cli.codigo = rec.cliente
        WHERE 
            rec.situacao IN ('A', 'B')

             -- Remove registros de revendas
            AND NOT EXISTS 
            (
                SELECT 1 
                FROM prd.dl_redeok_checkok.tb_revenda rev 
                WHERE rev.codigo = rec.cliente
            )

             -- Filtra os mesmos clientes da CTE 'clientes'
            AND cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')
            AND (cli.matriz = cli.codigo OR falsa = 's')
            AND cli.receita IN (9, 10, 46, 58)
            AND cli.cgcmf IS NOT NULL
            AND cli.cgcmf != '23236392000198'
            AND LENGTH(TRIM(cli.cgcmf)) = 14    
            AND cli.cgcmf RLIKE '^[0-9]+$'
    ) 
    GROUP BY 
        documento, cliente  
),

-- CTE para encontrar a data do último boleto gerado por cliente
ultimo_boleto AS 
(
    SELECT 
        cli.cgcmf AS documento,
        MAX(rec.emissao) AS ultima_emissao            -- Última data de emissão do boleto
    FROM 
        prd.dl_redeok_checkok.vw_receber rec
    INNER JOIN prd.dl_redeok_checkok.tb_cliente cli 
        ON cli.codigo = rec.cliente
    WHERE 
        rec.situacao IN ('A', 'B')                   -- Apenas boletos ativos ou em aberto

        -- Remove registros de revendas
        AND NOT EXISTS 
        (
            SELECT 1 
            FROM prd.dl_redeok_checkok.tb_revenda rev 
            WHERE rev.codigo = rec.cliente
        )
        AND cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')
        AND (cli.matriz = cli.codigo OR falsa = 's')
        AND cli.receita IN (9, 10, 46, 58)
        AND cli.cgcmf IS NOT NULL
        AND cli.cgcmf != '23236392000198'
        AND LENGTH(TRIM(cli.cgcmf)) = 14    
        AND cli.cgcmf RLIKE '^[0-9]+$'
    GROUP BY cli.cgcmf
),

-- CTE para encontrar a data do primeiro boleto gerado por cliente
primeiro_boleto AS 
(
    SELECT 
        cli.cgcmf AS documento,
        MIN(rec.emissao) AS data_primeiro_boleto
    FROM 
        prd.dl_redeok_checkok.vw_receber rec
    INNER JOIN prd.dl_redeok_checkok.tb_cliente cli 
        ON cli.codigo = rec.cliente
    WHERE 
        rec.situacao IN ('A', 'B')
        AND NOT EXISTS 
        (
            SELECT 1 
            FROM prd.dl_redeok_checkok.tb_revenda rev 
            WHERE rev.codigo = rec.cliente
        )
        AND cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')
        AND (cli.matriz = cli.codigo OR falsa = 's')
        AND cli.receita IN (9, 10, 46, 58)
        AND cli.cgcmf IS NOT NULL
        AND cli.cgcmf != '23236392000198'
        AND LENGTH(TRIM(cli.cgcmf)) = 14    
        AND cli.cgcmf RLIKE '^[0-9]+$'
    GROUP BY cli.cgcmf
),

-- CTE para somar o total pago por cliente
soma_valor_pago AS
(
    SELECT 
        cli.cgcmf AS documento,
        SUM(rec.valor_pago) AS total_valor_pago       -- Soma total de valores pagos pelo cliente
    FROM 
        prd.dl_redeok_checkok.vw_receber rec
    INNER JOIN prd.dl_redeok_checkok.tb_cliente cli 
        ON cli.codigo = rec.cliente
    WHERE 
        rec.situacao IN ('A', 'B')

        -- Remove registros de revendas
        AND NOT EXISTS 
        (
            SELECT 1 
            FROM prd.dl_redeok_checkok.tb_revenda rev 
            WHERE rev.codigo = rec.cliente
        )
        AND cli.sit_cliente IN ('A', 'F', 'N', 'P', 'D')
        AND (cli.matriz = cli.codigo OR falsa = 's')
        AND cli.receita IN (9, 10, 46, 58)
        AND cli.cgcmf IS NOT NULL
        AND cli.cgcmf != '23236392000198'
        AND LENGTH(TRIM(cli.cgcmf)) = 14    
        AND cli.cgcmf RLIKE '^[0-9]+$'
    GROUP BY cli.cgcmf
),
documentos_filtrados AS (
    SELECT 
          cli.contratos
        , cli.documento
        , tit.cliente  
        , cli.data_inicio
        , cli.data_final  
        , cli.max_conexao 
        , prim.data_primeiro_boleto   
        , CASE 
              WHEN cli.data_final IS NULL THEN NULL
              ELSE ult.ultima_emissao
          END AS data_ultimo_boleto
        , tit.boletos_gerados
        , ROUND(COALESCE(soma.total_valor_pago, 0), 1) AS total_valor_pago
        , cli.receita  -- Adicionando receita
        , ROW_NUMBER() OVER (PARTITION BY cli.documento ORDER BY tit.boletos_gerados DESC) AS row_number
     FROM 
        clientes cli
    INNER JOIN titulo_cobranca tit 
        ON cli.documento = tit.documento
    LEFT JOIN ultimo_boleto ult
        ON cli.documento = ult.documento
    LEFT JOIN primeiro_boleto prim
        ON cli.documento = prim.documento
    LEFT JOIN soma_valor_pago soma
        ON cli.documento = soma.documento
)
SELECT 
    contratos,
    documento,
    cliente,
    data_inicio,
    data_final,
    max_conexao,
    data_primeiro_boleto,
    data_ultimo_boleto,
    boletos_gerados,
    total_valor_pago,
    receita 
FROM documentos_filtrados
WHERE row_number = 1;

"""


# Executando a consulta e obtendo os dados em um DataFrame Spark
tabela_nova_sdf= spark.sql(query)

from pyspark.sql.functions import round

# Criar a coluna 'tipo_saida' com as condições especificadas
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "tipo_saida",
    F.when(F.col("max_conexao") == -1, "CANCELAMENTO NORMAL")
    .when(F.col("max_conexao") == 0, "90 DIAS BLOQUEADO")
    .when(F.col("max_conexao") == -5, "90 DIAS PRÉ CANCELADO")
    .when(F.col("max_conexao") == -7, "90 DIAS PRÉ CANCELADO JURÍDICO")
    .when(F.col("max_conexao").isNull(), "ATIVO")
    .otherwise("OUTROS")
)



from pyspark.sql.functions import col, to_date

from pyspark.sql.functions import col, to_timestamp

# Corrigir datas que possuem o ano como '0012' para '2012' e '0017' para '2017'
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "data_final", 
    regexp_replace("data_final", "^0011-", "2011-")  # Corrige 0012 para 2012
)


# Corrigir datas que possuem o ano como '0012' para '2012' e '0017' para '2017'
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "data_final", 
    regexp_replace("data_final", "^0201-", "2019-")  # Corrige 0012 para 2012
)

# Corrigir datas que possuem o ano como '0012' para '2012' e '0017' para '2017'
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "data_final", 
    regexp_replace("data_final", "^0012-", "2012-")  # Corrige 0012 para 2012
)

tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "data_final", 
    regexp_replace("data_final", "^0017-", "2017-")  # Corrige 0017 para 2017
)

# Adicionar o cálculo do meses_contrato tratando data_final nulo
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "meses_contrato",
    round(
        months_between(
            # Usa a data atual quando data_final for nulo
            coalesce(col("data_final"), current_date()),
            col("data_inicio")
        )
    )
)



tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "meses_boleto",
    F.ceil(
        F.months_between(
            F.coalesce(F.col("data_ultimo_boleto"), F.current_date()), 
            F.col("data_primeiro_boleto")
        )
    )
)



# Criar a coluna 'anos_boleto' dividindo 'meses_boleto' por 12 e arredondando para 2 casas decimais
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "anos_boleto",
    F.round(F.col("meses_boleto") / 12, 2)  # Arredonda para 2 casas decimais
)


tabela_nova_sdf = tabela_nova_sdf.withColumn("anos_contrato", round(col("meses_contrato") / 12, 2))

# Criar a coluna 'boleto_gerados_anos' dividindo 'boletos_gerados' por 12 e arredondando para 2 casas decimais
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "boleto_gerados_anos",
    F.round(F.col("boletos_gerados") / 12, 2)  # Arredonda para 2 casas decimais
)

tabela_nova_sdf = tabela_nova_sdf.withColumn("churn",when(col("data_final").isNotNull(), 1).otherwise(0))

tabela_nova_sdf = tabela_nova_sdf.withColumn("ano_inicio", year(col("data_inicio"))) 



tabela_nova_sdf = tabela_nova_sdf.filter(col("contratos") == 1)

# Substituir valores nulos pela data atual
# Criar colunas corrigindo a diferença para ser sempre positiva
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "diferenca_meses_inicio_primeiro_boleto",
    expr("ceil(abs(months_between(data_primeiro_boleto, data_inicio)))")
).withColumn(
    "diferenca_meses_final_ultimo_boleto",
    expr("abs(floor(months_between(COALESCE(data_final, current_date()), COALESCE(data_ultimo_boleto, current_date()))))")
)

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, expr

# Criar a classificação para diferença de meses
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "classifica_diferenca_boleto_tempo_inicial",
    when(col("diferenca_meses_inicio_primeiro_boleto") <= 1, "0 a 1 mês")
    .when((col("diferenca_meses_inicio_primeiro_boleto") >= 2) & (col("diferenca_meses_inicio_primeiro_boleto") <= 5), "2 a 5 meses")
    .when((col("diferenca_meses_inicio_primeiro_boleto") >= 6) & (col("diferenca_meses_inicio_primeiro_boleto") <= 12), "6 a 12 meses")
    .when((col("diferenca_meses_inicio_primeiro_boleto") > 12) & (col("diferenca_meses_inicio_primeiro_boleto") <= 24), "13 a 24 meses")
    .otherwise("Maior que 24 meses")
).withColumn(
    "classifica_diferenca_boleto_tempo_final",
    when(col("diferenca_meses_final_ultimo_boleto") <= 1, "0 a 1 mês")
    .when((col("diferenca_meses_final_ultimo_boleto") >= 2) & (col("diferenca_meses_final_ultimo_boleto") <= 5), "2 a 5 meses")
    .when((col("diferenca_meses_final_ultimo_boleto") >= 6) & (col("diferenca_meses_final_ultimo_boleto") <= 12), "6 a 12 meses")
    .when((col("diferenca_meses_final_ultimo_boleto") > 12) & (col("diferenca_meses_final_ultimo_boleto") <= 24), "13 a 24 meses")
    .otherwise("Maior que 24 meses")
)

tabela_nova_sdf = tabela_nova_sdf.filter((col("meses_contrato") > 0))

# Pegando o ano das datas
#df = tabela_nova_sdf.withColumn("ano_primeiro_boleto", year(col("data_primeiro_boleto"))) \
       #.withColumn("ano_ultimo_boleto", year(col("data_ultimo_boleto")))

tabela_nova_sdf = tabela_nova_sdf.filter(
    (year(col("data_primeiro_boleto")).isNotNull()) & 
    (year(col("data_primeiro_boleto")) >= 1900) & 
    ((col("data_ultimo_boleto").isNull()) | (year(col("data_ultimo_boleto")) >= 1900)) &  
    (year(col("data_primeiro_boleto")) <= 2100) & 
    ((col("data_ultimo_boleto").isNull()) | (year(col("data_ultimo_boleto")) <= 2100))
)





In [6]:
# Realizar o join entre tabela_nova_sdf e resultado_df usando documento = CNPJ_CPF
tabela_nova_sdf = tabela_nova_sdf.join(resultado_df, tabela_nova_sdf["documento"] == resultado_df["CNPJ_CPF"], "left")

In [7]:
from pyspark.sql.functions import to_timestamp

tabela_nova_sdf = tabela_nova_sdf.withColumn("data_inicio", to_date(tabela_nova_sdf["data_inicio"], "yyyy-MM-dd HH:mm:ss"))
tabela_nova_sdf = tabela_nova_sdf.withColumn("data_final", to_date(tabela_nova_sdf["data_final"], "yyyy-MM-dd HH:mm:ss"))
tabela_nova_sdf = tabela_nova_sdf.withColumn("data_primeiro_boleto", to_date(tabela_nova_sdf["data_primeiro_boleto"], "yyyy-MM-dd HH:mm:ss"))
tabela_nova_sdf = tabela_nova_sdf.withColumn("data_ultimo_boleto", to_date(tabela_nova_sdf["data_ultimo_boleto"], "yyyy-MM-dd HH:mm:ss"))

In [8]:
from pyspark.sql.functions import when

# Atualizando a coluna meses_boleto, substituindo 0 por 1
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "meses_boleto",
    when(tabela_nova_sdf["meses_boleto"] == 0, 1).otherwise(tabela_nova_sdf["meses_boleto"])
)


# Atualizando a coluna meses_boleto, substituindo 0 por 1
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "anos_boleto",
    when(tabela_nova_sdf["anos_boleto"] == 0, 0.08).otherwise(tabela_nova_sdf["anos_boleto"])
)


# Criando a coluna 'empresa' baseada nos valores de 'receita'
tabela_nova_sdf = tabela_nova_sdf.withColumn(
    "empresa",
    F.when(F.col("receita").isin(58, 9), "RedeOk")
     .when(F.col("receita") == 46, "Autofax")
     .when(F.col("receita") == 10, "Express")
     .otherwise("Desconhecido")  # Caso haja outros valores não mapeados
)

In [9]:
tabela_nova_sdf.show()

+---------+--------------+-------+-----------+----------+-----------+--------------------+------------------+---------------+----------------+-------+-------------------+--------------+------------+-----------+-------------+-------------------+-----+----------+--------------------------------------+-----------------------------------+-----------------------------------------+---------------------------------------+--------------+-------+--------------------+----+---------------------+-------+
|contratos|     documento|cliente|data_inicio|data_final|max_conexao|data_primeiro_boleto|data_ultimo_boleto|boletos_gerados|total_valor_pago|receita|         tipo_saida|meses_contrato|meses_boleto|anos_boleto|anos_contrato|boleto_gerados_anos|churn|ano_inicio|diferenca_meses_inicio_primeiro_boleto|diferenca_meses_final_ultimo_boleto|classifica_diferenca_boleto_tempo_inicial|classifica_diferenca_boleto_tempo_final|      CNPJ_CPF|CNAE_RF|          Hierarquia|  uf|data_inicio_atividade|empresa|
+---

In [10]:
tabela_nova_sdf.count()

130225

In [11]:
df_pandas = tabela_nova_sdf.toPandas()

In [12]:
df_pandas.to_csv('tabela_sobrevivencia_4.csv', index=False, encoding='utf-8')

In [14]:
tabela_nova_df = pd.read_csv(r"C:\Users\gusta\OneDrive\Área de Trabalho\survivor\tabela_sobrevivencia_3.csv")

In [16]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [17]:
display(tabela_nova_df)

,contratos,documento,cliente,data_inicio,data_final,max_conexao,data_primeiro_boleto,data_ultimo_boleto,boletos_gerados,total_valor_pago,receita,tipo_saida,meses_contrato,meses_boleto,anos_boleto,anos_contrato,boleto_gerados_anos,churn,ano_inicio,diferenca_meses_inicio_primeiro_boleto,diferenca_meses_final_ultimo_boleto,classifica_diferenca_boleto_tempo_inicial,classifica_diferenca_boleto_tempo_final,CNPJ_CPF,CNAE_RF,Hierarquia,uf,data_inicio_atividade
0,1,750329882,457047,2010-11-30,2018-05-02,0.0,2017-10-01,2018-05-01,8,452.6,10,90 DIAS BLOQUEADO,89.0,7,0.58,7.42,0.67,1,2010,83,0,Maior que 24 meses,0 a 1 mês,NaN,NaN,NaN,NaN,NaN
1,1,17880000109,194574,2007-04-23,2009-08-31,-1.0,2007-05-01,2009-09-01,28,1541.6,9,CANCELAMENTO NORMAL,28.0,28,2.33,2.33,2.33,1,2007,1,1,0 a 1 mês,0 a 1 mês,1.788000e+10,4744005.0,Comércio; Comércio varejista; Materiais de con...,SP,19940614.0
2,1,30277000167,224704,2009-02-06,2010-06-05,-3.0,2009-03-01,2012-12-06,14,207.5,9,OUTROS,16.0,46,3.83,1.33,1.17,1,2009,1,31,0 a 1 mês,Maior que 24 meses,3.027700e+10,1013902.0,Indústria; Indústrias de Transformação; Alimen...,SP,19940824.0
3,1,78792000117,224933,2009-01-23,2010-01-14,-1.0,2009-02-01,2012-12-06,9,141.7,9,CANCELAMENTO NORMAL,12.0,47,3.92,1.00,0.75,1,2009,1,35,0 a 1 mês,Maior que 24 meses,7.879200e+10,4921301.0,Serviços; Transporte; Transporte terrestre,PR,19940530.0
4,1,102401000152,27915,2002-03-18,2008-03-31,-1.0,2006-02-01,2008-05-01,15,353.4,9,CANCELAMENTO NORMAL,72.0,27,2.25,6.00,1.25,1,2002,47,2,Maior que 24 meses,2 a 5 meses,1.024010e+11,4731800.0,Comércio; Comércio varejista; Combustíveis,GO,19940630.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130220,1,97340087000188,367791,2015-03-11,2015-07-07,-1.0,2015-04-01,2015-07-01,4,240.0,58,CANCELAMENTO NORMAL,4.0,3,0.25,0.33,0.33,1,2015,1,0,0 a 1 mês,0 a 1 mês,9.734009e+13,2599399.0,Indústria; Indústrias de Transformação; Metais,PR,19940331.0
130221,1,97427199000170,357714,2014-10-08,2016-08-31,-1.0,2014-11-01,2016-08-01,22,26543.9,58,CANCELAMENTO NORMAL,23.0,21,1.75,1.92,1.83,1,2014,1,0,0 a 1 mês,0 a 1 mês,9.742720e+13,1112700.0,Indústria; Indústrias de Transformação; Alimen...,SC,19940418.0
130222,1,97519260000100,324879,2013-07-02,2016-10-28,-1.0,2013-08-01,2016-10-01,39,1916.3,58,CANCELAMENTO NORMAL,40.0,38,3.17,3.33,3.25,1,2013,1,0,0 a 1 mês,0 a 1 mês,9.751926e+13,6190601.0,Serviços; Informação e comunicação; Telecomuni...,MS,20110708.0
130223,1,97535278000103,336978,2014-01-09,2014-04-05,-3.0,2014-06-06,2014-06-06,1,0.0,9,OUTROS,3.0,1,0.08,0.25,0.08,1,2014,5,3,2 a 5 meses,2 a 5 meses,9.753528e+13,4754701.0,Comércio; Comércio varejista; Móveis,MT,20110712.0


In [14]:
ltv = spark.table("dev.dw_rok.tb_ltv") 

In [15]:
ltv.show()

+---------+--------------+-------+-----------+----------+-----------+--------------------+------------------+---------------+----------------+-------+-------------------+--------------+------------+-----------+-------------+-------------------+-----+----------+--------------------------------------+-----------------------------------+-----------------------------------------+---------------------------------------+--------------+-------+--------------------+----+---------------------+-------+
|contratos|     documento|cliente|data_inicio|data_final|max_conexao|data_primeiro_boleto|data_ultimo_boleto|boletos_gerados|total_valor_pago|receita|         tipo_saida|meses_contrato|meses_boleto|anos_boleto|anos_contrato|boleto_gerados_anos|churn|ano_inicio|diferenca_meses_inicio_primeiro_boleto|diferenca_meses_final_ultimo_boleto|classifica_diferenca_boleto_tempo_inicial|classifica_diferenca_boleto_tempo_final|      CNPJ_CPF|CNAE_RF|          Hierarquia|  uf|data_inicio_atividade|empresa|
+---

In [16]:
ltv.count()

130225

In [18]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from lifelines import KaplanMeierFitter, WeibullFitter, CoxPHFitter

def analisar_sobrevivencia_comparada_boletos(df):
    resultados = {"Ano": [], "Kaplan-Meier": [], "Weibull": [], "Cox": []}
    
    df["data_inicio"] = pd.to_datetime(df["data_inicio"])
    df["data_primeiro_boleto"] = pd.to_datetime(df["data_primeiro_boleto"])
    
    for ano_filtro in range(2006, 2025):
        print(f"Agora rodando para o ano {ano_filtro}...")
        
        
        df_filtrado = df[(df['data_primeiro_boleto'].dt.year >= ano_filtro)]
        
        df_filtrado["anos_boleto"] = df_filtrado["anos_boleto"].astype(float)
        df_filtrado["churn"] = df_filtrado["churn"].astype(int)
        
        if df_filtrado.empty:
            continue
        
        resultados["Ano"].append(ano_filtro)
        
        # Kaplan-Meier
        kmf = KaplanMeierFitter()
        kmf.fit(df_filtrado["anos_boleto"], event_observed=df_filtrado["churn"])
        tempo_km = np.trapz(kmf.survival_function_["KM_estimate"], kmf.survival_function_.index)
        resultados["Kaplan-Meier"].append(tempo_km)
        
        # Weibull
        wf = WeibullFitter()
        wf.fit(df_filtrado["anos_boleto"], event_observed=df_filtrado["churn"])
        tempo_weibull = wf.lambda_ * np.math.gamma(1 + 1 / wf.rho_)
        resultados["Weibull"].append(tempo_weibull)
        
        # Cox
        colunas_cox = ["anos_boleto", "churn", "total_valor_pago", "boletos_gerados", "ano_inicio"]
        if all(col in df_filtrado.columns for col in colunas_cox):
            df_cox = df_filtrado[colunas_cox].dropna()
            if not df_cox.empty:
                cph = CoxPHFitter()
                cph.fit(df_cox, duration_col="anos_boleto", event_col="churn")
                tempo_cox = cph.predict_expectation(df_cox).mean()
                resultados["Cox"].append(tempo_cox)
            else:
                resultados["Cox"].append(None)
        else:
            resultados["Cox"].append(None)
    
    df_resultados = pd.DataFrame(resultados)
    
    # Plotando gráfico
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_resultados["Ano"].astype(str), y=df_resultados["Kaplan-Meier"],
                             mode="lines+markers", name="Kaplan-Meier", line=dict(color="blue")))
    fig.add_trace(go.Scatter(x=df_resultados["Ano"].astype(str), y=df_resultados["Weibull"],
                             mode="lines+markers", name="Weibull", line=dict(color="red")))
    fig.add_trace(go.Scatter(x=df_resultados["Ano"].astype(str), y=df_resultados["Cox"],
                             mode="lines+markers", name="Cox", line=dict(color="green")))
    fig.update_layout(
        title="Comparação do Tempo de Vida Esperado (Kaplan-Meier, Weibull, Cox) Boletos",
        xaxis_title="Ano de Início",
        yaxis_title="Tempo de Vida Esperado (anos)",
        template="plotly_white",
        legend_title="Modelos"
    )
    fig.show()
    
    return df_resultados


In [19]:
analisar_sobrevivencia_comparada_boletos(tabela_nova_df)

Agora rodando para o ano 2006...


C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado["anos_boleto"] = df_filtrado["anos_boleto"].astype(float)
C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado["churn"] = df_filtrado["churn"].astype(int)
C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:35: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (D

Agora rodando para o ano 2007...


C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado["anos_boleto"] = df_filtrado["anos_boleto"].astype(float)
C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado["churn"] = df_filtrado["churn"].astype(int)
C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:35: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (D

Agora rodando para o ano 2008...


...

Agora rodando para o ano 2021...
Agora rodando para o ano 2022...


C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado["anos_boleto"] = df_filtrado["anos_boleto"].astype(float)
C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado["churn"] = df_filtrado["churn"].astype(int)
C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:35: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (D

Agora rodando para o ano 2023...
Agora rodando para o ano 2024...


C:\Users\gusta\AppData\Local\Temp\ipykernel_19616\573334061.py:35: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (Deprecated Numpy 1.25). Replace usages of `np.math` with `math`
  tempo_weibull = wf.lambda_ * np.math.gamma(1 + 1 / wf.rho_)


,Ano,Kaplan-Meier,Weibull,Cox
0,2006,3.722740,3.803778,3.414830
1,2007,3.720341,3.818884,3.408379
2,2008,3.668760,3.784276,3.329890
3,2009,3.585593,3.713838,3.247633
4,2010,3.537666,3.670082,3.186463
5,2011,3.425692,3.580448,3.084897
6,2012,3.303445,3.487252,2.953041
7,2013,3.246747,3.461976,2.873800
8,2014,3.190979,3.465747,2.800250
9,2015,3.122819,3.463787,2.706414
